### Some studies for the dwpt

In [ ]:
import sys
sys.path.append("..")

In [ ]:
import matplotlib.pyplot as plt
import numpy as np

import wavelet_transform.wpt.transform as wpt
import wavelet_transform.wpt.inverse_transform as inv
from wavelet_transform.utils.extensions import fft
import wavelet_transform.utils.PCD_data as pcd
from wavelet_transform.utils.preprocessing import preprocess

### Frequency of nodes
Testing the validity of the assigned frequency range of nodes with synthetic signal

In [ ]:
duration = 30
sampling_rate = 10
n_samples = int(duration * sampling_rate)

freq_spectrum = np.ones(n_samples, dtype=complex)
signal = np.fft.ifft(freq_spectrum)
signal = np.abs(signal)
t = np.linspace(0, 10, n_samples)

plt.plot(t, signal)
plt.title("Signal")
plt.xlabel("t [s]")
plt.ylabel("Amplitude")
plt.show()

fft_results, frequencies = fft(signal, sampling_rate)
plt.stem(frequencies, np.abs(fft_results))
plt.show()


In [ ]:
levels = 7
transform = wpt.dwpt(signal, levels, "haar")
nodes = transform.get_level(order="freq")  # get max level
peak_freqs = []
cent_freqs = []
avg_freqs = []
for node in nodes:
    rec = inv.inv_dwpt(transform, nodes=node)
    fft_results, frequencies = fft(rec, sampling_rate)
    fft_results = abs(fft_results)

    norm_fft_results = fft_results / sum(fft_results)
    avg_freq = sum(frequencies * norm_fft_results)
    cent_freq = 1 / 2 * sum(node.numeric_freq_range) * sampling_rate / 2
    freq_peak = frequencies[np.argmax(fft_results)]

    peak_freqs.append(freq_peak)
    avg_freqs.append(avg_freq)    
    cent_freqs.append(cent_freq)

    # print(
    #     f"""
    #     node centre - {cent_freq:.2f} Hz
    #     avg freq    - {avg_freq:.2f} Hz
    #     peak freq   - {freq_peak:.2e} Hz
    #     """
    # )

    # plt.stem(frequencies, np.abs(fft_results))
    # plt.title(f"fft for node {node.path}")
    # plt.ylabel('Magnitude')
    # plt.xlabel("Frequency [Hz]")
    # plt.show()

plt.plot(cent_freqs, peak_freqs, label="peak frequency")
plt.plot(cent_freqs, avg_freqs, label="average frequency")
plt.plot(cent_freqs, cent_freqs, label="centre frequency", linestyle="--")
plt.title("Frequencies vs centre frequency of node")
plt.ylabel("Frequency [Hz]")
plt.xlabel("Centre frequency of node [Hz]")
plt.legend()
plt.show()

### Reconstruction error
Testing reconstruction error vs frequency threshold with pcd data

In [ ]:
max_level = 7

signal_pcd, t_plot = pcd.get_data((280, 1))
signal_pcd = preprocess(signal_pcd, window=True)
sampling_rate = 200e6
transform = wpt.dwpt(signal_pcd, max_level, "db10")

freq_thresholds = []
avg_errors = []

for freq_threshold in [i for i in np.linspace(0, 1, 200)]:
    rec = np.zeros(len(signal_pcd))
    leaf_nodes = transform.get_leaf_nodes(freq_threshold=freq_threshold)
    if len(leaf_nodes) == 0:
        continue
    for node in leaf_nodes:
        rec += inv.inv_dwpt(transform, nodes=node.path)
    total_error = sum(signal_pcd - rec) ** 2 / len(rec)
    avg_error = np.average((signal_pcd - rec) ** 2)
    freq_thresholds.append(freq_threshold * sampling_rate / 2)
    avg_errors.append(avg_error)


In [ ]:
plt.plot(freq_thresholds, avg_errors)
plt.title("Reconstruction error vs frequency threshold")
plt.ylabel("Average reconstruction error")
plt.xlabel("Frequency [Hz]")
plt.show()
print(avg_errors)

### Statistical metrics of wavelet coeffs vs reconstructed coeffs
Plotting the standardised statistical metrics calculated directly on wavelet coefficients and on their full reconstructions. 

In [ ]:
max_level = 10
min_entropy = True
min_level = 6
freq_threshold = 5e6 / 100e6
transform = wpt.dwpt(
    signal_pcd, 
    max_level, 
    "sym20", 
    min_entropy=min_entropy, 
    min_level=min_level,
    freq_threshold=freq_threshold
)
inv_transform = inv.InverseTransform(transform)

dir_results = transform.info(leaf_nodes=True, order="freq")
inv_results = inv_transform.info(leaf_nodes=True, order="freq")
print(dir_results.columns.tolist())

In [ ]:
metric = "energy"  # metric to plot
scaling = "min"  # standard or min

def get_result(df, level, node_number):
    n = node_number
    dir_row = df[(df["level"] == level) & (df["node number"] == n)]
    return dir_row[metric].values[0]

def def_scaler(mode):
    def scale(array, mode=mode):
        if mode == "standard":
            return (array - np.average(array)) / np.std(array)
        elif mode == "min":
            return array / min(array)
    return scale

dir_results_plot = []
inv_results_plot = []
levels = []
frequencies = []

for node in transform.get_leaf_nodes():
    level = node.level
    n = node.node_number

    dir_results_plot.append(get_result(dir_results, level, n))
    inv_results_plot.append(get_result(inv_results, level, n))
    levels.append(level)

    centre_freq = 1 / 2 * sum(node.numeric_freq_range)
    frequencies.append(centre_freq * sampling_rate / 2)

scale = def_scaler(scaling)
dir_results_plot = scale(dir_results_plot)
inv_results_plot = scale(inv_results_plot)

fig, ax = plt.subplots()
ax2 = ax.twinx()
ax.scatter(frequencies, dir_results_plot, label="Wavelet coefficients")
ax.scatter(frequencies, inv_results_plot, label="Reconstructions", marker="x")
ax2.scatter(frequencies, levels, label="Level", marker="+", color="green")

ax.set_xlabel(f"Frequency [Hz]")
ax.set_ylabel("Value")
ax.set_title(f"{metric.capitalize()} of nodes")
ax.grid(True)
ax2.set_ylabel("Level")
fig.legend()
plt.show()